# 02a: COMPAS Data Cleaning

**Purpose:** Apply data cleaning and validation to prepare COMPAS data for modeling

**Dataset:** ProPublica COMPAS data (post-EDA)

**Author:** TabPFN for Criminology Research Team

**Date:** 2025-11-08

---

## Overview

### Purpose
This notebook performs data cleaning and validation:
1. Load raw COMPAS data
2. Apply filtering criteria from analysis plan
3. Validate data quality
4. Handle any edge cases
5. Save cleaned data for feature engineering

### Inputs
- **Data:** Raw COMPAS data from ProPublica
- **Prior notebooks:** `01a_compas_eda.ipynb` (findings inform decisions)
- **Analysis plan:** `docs/methodology/analysis_plan.md` (filtering criteria)

### Outputs
- Cleaned dataset → `data/processed/compas_cleaned.parquet`
- Cleaning report → `data/metadata/compas_cleaning_log.json`
- Filtering flowchart data → `results/tables/filtering_flowchart.csv`

### Runtime
**Expected:** 1-2 minutes

---

## 1. Setup

In [1]:
# Standard library
import sys
from pathlib import Path
import json
from datetime import datetime

# Add src to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / "src"))

# Data manipulation
import numpy as np
import pandas as pd

# Our modules
from data.compas_loader import COMPASDataLoader

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

# Random seed
np.random.seed(42)

print("✓ Setup complete")

✓ Setup complete


In [2]:
# Directories
DATA_DIR = project_root / "data"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"
TABLES_DIR = project_root / "results" / "tables"

# Create directories
for dir_path in [PROCESSED_DIR, METADATA_DIR, TABLES_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("✓ Directories configured")

✓ Directories configured


---

## 2. Load Data

Load raw COMPAS data using the data loader (same as EDA).

In [3]:
# Initialize loader
loader = COMPASDataLoader(data_dir=DATA_DIR / "raw" / "compas")

# Load data
print("Loading COMPAS data...")
data_dict = loader.load_and_prepare()

df_raw = data_dict['data']
metadata = data_dict['metadata']
sensitive_attrs = data_dict['sensitive_features']

print(f"✓ Raw data loaded: {len(df_raw):,} samples, {len(df_raw.columns)} features")

Loading COMPAS data...
✓ Raw data loaded: 6,172 samples, 12 features


---

## 3. Data Cleaning

### 3.1 Filtering Criteria

Per analysis plan (`docs/methodology/analysis_plan.md`), apply:

**Inclusion Criteria:**
- Days between arrest and COMPAS screening ≤ 30 (already applied by loader)
- Valid charge degree (M or F) (already applied)
- Non-missing recidivism outcome (already applied)
- Non-missing demographics (race, sex, age) (already applied)

**Note:** The COMPASDataLoader already applies ProPublica's standard filtering, so most cleaning is done. This notebook documents and validates that cleaning.

In [11]:
# Track filtering stages
filtering_stages = []

# Initial raw data
filtering_stages.append({
    'stage': '0_raw_data',
    'description': 'Raw ProPublica COMPAS data',
    'n_samples': len(df_raw),
    'n_removed': 0
})

print(f"Starting samples: {len(df_raw):,}")

Starting samples: 6,172


### 3.2 Check for Missing Values

In [8]:
# Combine with sensitive attributes for complete check
df_full = df_raw

# Check missing values
missing_any = df_full.isnull().any().any()

if missing_any:
    print("⚠ Missing values detected:")
    missing_summary = df_full.isnull().sum()
    missing_summary = missing_summary[missing_summary > 0]
    display(missing_summary)
    
    # Remove rows with any missing values
    n_before = len(df_full)
    df_full = df_full.dropna()
    n_after = len(df_full)
    n_removed = n_before - n_after
    
    filtering_stages.append({
        'stage': '1_remove_missing',
        'description': 'Remove rows with missing values',
        'n_samples': n_after,
        'n_removed': n_removed
    })
    
    print(f"Removed {n_removed:,} rows with missing values")
else:
    print("✓ No missing values detected")
    filtering_stages.append({
        'stage': '1_remove_missing',
        'description': 'Check for missing values',
        'n_samples': len(df_full),
        'n_removed': 0
    })

✓ No missing values detected


### 3.3 Check for Duplicates

In [9]:
# Check duplicates
n_duplicates = df_full.duplicated().sum()

if n_duplicates > 0:
    print(f"⚠ Found {n_duplicates:,} duplicate rows")
    
    # Remove duplicates (keep first occurrence)
    n_before = len(df_full)
    df_full = df_full.drop_duplicates(keep='first')
    n_after = len(df_full)
    n_removed = n_before - n_after
    
    filtering_stages.append({
        'stage': '2_remove_duplicates',
        'description': 'Remove duplicate rows (keep first)',
        'n_samples': n_after,
        'n_removed': n_removed
    })
    
    print(f"Removed {n_removed:,} duplicate rows")
else:
    print("✓ No duplicate rows")
    filtering_stages.append({
        'stage': '2_remove_duplicates',
        'description': 'Check for duplicates',
        'n_samples': len(df_full),
        'n_removed': 0
    })

⚠ Found 963 duplicate rows
Removed 963 duplicate rows


### 3.4 Validate Data Types

In [10]:
# Expected data types
target_col = metadata['target']

# Validate target is binary
unique_target = df_full[target_col].unique()
print(f"Target variable '{target_col}' unique values: {sorted(unique_target)}")

if set(unique_target) != {0, 1}:
    raise ValueError(f"Target variable must be binary (0, 1), found: {unique_target}")
else:
    print("✓ Target variable is binary")

# Validate categorical variables
categorical_cols = ['race', 'sex', 'age_cat']
for col in categorical_cols:
    if col in df_full.columns:
        print(f"\n{col}: {df_full[col].nunique()} categories")
        print(f"  {list(df_full[col].unique())}")

Target variable 'two_year_recid' unique values: [np.int64(0), np.int64(1)]
✓ Target variable is binary

race: 6 categories
  ['Other', 'African-American', 'Caucasian', 'Hispanic', 'Asian', 'Native American']

sex: 2 categories
  ['Male', 'Female']

age_cat: 3 categories
  ['Greater than 45', '25 - 45', 'Less than 25']


### 3.5 Validate Value Ranges

Check that continuous features have sensible ranges.

In [12]:
# Identify continuous features
continuous_cols = df_full.select_dtypes(include=[np.number]).columns.tolist()
if target_col in continuous_cols:
    continuous_cols.remove(target_col)

# Check for negative values where not expected
count_features = [col for col in continuous_cols if 'count' in col.lower() or 'priors' in col.lower()]

print("Checking count features for negative values:")
issues_found = False
for col in count_features:
    if col in df_full.columns:
        min_val = df_full[col].min()
        if min_val < 0:
            print(f"  ⚠ {col}: min = {min_val} (negative!)")
            issues_found = True
        else:
            print(f"  ✓ {col}: min = {min_val}, max = {df_full[col].max()}")

if not issues_found:
    print("\n✓ All count features have valid ranges")

Checking count features for negative values:
  ✓ juv_fel_count: min = 0, max = 20
  ✓ juv_misd_count: min = 0, max = 13
  ✓ juv_other_count: min = 0, max = 9
  ✓ priors_count: min = 0, max = 38

✓ All count features have valid ranges


---

## 4. Data Validation

### 4.1 Sample Size Check

In [13]:
# Check final sample size
n_final = len(df_full)
n_initial = filtering_stages[0]['n_samples']
n_total_removed = n_initial - n_final

print(f"Sample size summary:")
print(f"  Initial:  {n_initial:,}")
print(f"  Removed:  {n_total_removed:,} ({n_total_removed/n_initial*100:.1f}%)")
print(f"  Final:    {n_final:,}")

# Check against analysis plan requirement (adequate power)
MIN_SAMPLE_SIZE = 1000  # Per analysis plan, need adequate power
if n_final < MIN_SAMPLE_SIZE:
    print(f"\n⚠ WARNING: Sample size ({n_final:,}) below minimum ({MIN_SAMPLE_SIZE:,})")
else:
    print(f"\n✓ Sample size adequate ({n_final:,} ≥ {MIN_SAMPLE_SIZE:,})")

Sample size summary:
  Initial:  6,172
  Removed:  963 (15.6%)
  Final:    5,209

✓ Sample size adequate (5,209 ≥ 1,000)


### 4.2 Class Balance Check

In [14]:
# Check class distribution
class_dist = df_full[target_col].value_counts()
class_prop = df_full[target_col].value_counts(normalize=True)

print("Class distribution after cleaning:")
print(f"  No recidivism (0): {class_dist[0]:,} ({class_prop[0]:.1%})")
print(f"  Recidivism (1):    {class_dist[1]:,} ({class_prop[1]:.1%})")
print(f"\nBase rate: {class_prop[1]:.1%}")
print(f"Imbalance ratio: {class_dist[0] / class_dist[1]:.2f}:1")

# Check if imbalance is severe
imbalance_ratio = max(class_dist) / min(class_dist)
if imbalance_ratio > 3:
    print(f"\n⚠ Severe class imbalance (ratio {imbalance_ratio:.1f}:1)")
    print("   Consider: class weighting, SMOTE, or stratified sampling")
elif imbalance_ratio > 1.5:
    print(f"\n✓ Moderate class imbalance (ratio {imbalance_ratio:.1f}:1)")
    print("   Recommendation: Use class weighting and AUPRC metric")
else:
    print(f"\n✓ Classes well balanced (ratio {imbalance_ratio:.1f}:1)")

Class distribution after cleaning:
  No recidivism (0): 2,665 (51.2%)
  Recidivism (1):    2,544 (48.8%)

Base rate: 48.8%
Imbalance ratio: 1.05:1

✓ Classes well balanced (ratio 1.0:1)


### 4.3 Group Size Check

Verify we have adequate samples in each demographic group for fairness analysis.

In [15]:
# Check group sizes
MIN_GROUP_SIZE = 50  # Minimum for reliable statistical analysis

print("Group sizes for fairness analysis:")
print("\nRace:")
race_counts = df_full['race'].value_counts()
for race, count in race_counts.items():
    status = "✓" if count >= MIN_GROUP_SIZE else "⚠"
    print(f"  {status} {race}: {count:,}")

print("\nSex:")
sex_counts = df_full['sex'].value_counts()
for sex, count in sex_counts.items():
    status = "✓" if count >= MIN_GROUP_SIZE else "⚠"
    print(f"  {status} {sex}: {count:,}")

print("\nAge Category:")
age_counts = df_full['age_cat'].value_counts()
for age, count in age_counts.items():
    status = "✓" if count >= MIN_GROUP_SIZE else "⚠"
    print(f"  {status} {age}: {count:,}")

# Check if any group too small
all_groups = pd.concat([race_counts, sex_counts, age_counts])
if (all_groups < MIN_GROUP_SIZE).any():
    small_groups = all_groups[all_groups < MIN_GROUP_SIZE]
    print(f"\n⚠ WARNING: {len(small_groups)} group(s) below minimum size ({MIN_GROUP_SIZE})")
    print("   Fairness analysis may have limited power for these groups")
else:
    print(f"\n✓ All groups adequate for fairness analysis (≥ {MIN_GROUP_SIZE})")

Group sizes for fairness analysis:

Race:
  ✓ African-American: 2,694
  ✓ Caucasian: 1,703
  ✓ Hispanic: 469
  ✓ Other: 301
  ⚠ Asian: 31
  ⚠ Native American: 11

Sex:
  ✓ Male: 4,165
  ✓ Female: 1,044

Age Category:
  ✓ 25 - 45: 3,051
  ✓ Greater than 45: 1,097
  ✓ Less than 25: 1,061

⚠ WARNING: 2 group(s) below minimum size (50)
   Fairness analysis may have limited power for these groups


---

## 5. Save Cleaned Data

### 5.1 Separate Features and Target

In [16]:
# Separate sensitive attributes
sensitive_cols = ['race', 'sex', 'age_cat']
feature_cols = [col for col in df_full.columns if col not in [target_col] + sensitive_cols]

# Create separate dataframes
df_features = df_full[feature_cols].copy()
df_target = df_full[target_col].copy()
df_sensitive = df_full[sensitive_cols].copy()

print(f"Features: {len(feature_cols)} columns")
print(f"Target: {target_col}")
print(f"Sensitive attributes: {sensitive_cols}")

Features: 8 columns
Target: two_year_recid
Sensitive attributes: ['race', 'sex', 'age_cat']


### 5.2 Save to Parquet Format

In [17]:
# Save cleaned full dataset
output_path = PROCESSED_DIR / "compas_cleaned.parquet"
df_full.to_parquet(output_path, index=False)
print(f"✓ Saved cleaned data: {output_path}")
print(f"  Size: {output_path.stat().st_size / 1024:.1f} KB")

# Also save as components for flexibility
df_features.to_parquet(PROCESSED_DIR / "compas_features.parquet", index=False)
df_target.to_frame().to_parquet(PROCESSED_DIR / "compas_target.parquet", index=False)
df_sensitive.to_parquet(PROCESSED_DIR / "compas_sensitive.parquet", index=False)

print("\n✓ Saved component files:")
print("  - compas_features.parquet")
print("  - compas_target.parquet")
print("  - compas_sensitive.parquet")

✓ Saved cleaned data: /storage/work/szn5432/TabPFN-for-Criminology/data/processed/compas_cleaned.parquet
  Size: 28.9 KB

✓ Saved component files:
  - compas_features.parquet
  - compas_target.parquet
  - compas_sensitive.parquet


### 5.3 Save Cleaning Log

In [18]:
# Create comprehensive cleaning log
cleaning_log = {
    'notebook': '02a_data_cleaning.ipynb',
    'execution_date': datetime.now().isoformat(),
    'dataset': 'COMPAS',
    'filtering_stages': filtering_stages,
    'initial_samples': n_initial,
    'final_samples': n_final,
    'samples_removed': n_total_removed,
    'removal_percentage': round(n_total_removed / n_initial * 100, 2),
    'features': {
        'n_features': len(feature_cols),
        'feature_names': feature_cols,
        'target': target_col,
        'sensitive_attributes': sensitive_cols
    },
    'class_distribution': {
        'positive': int(class_dist[1]),
        'negative': int(class_dist[0]),
        'base_rate': float(class_prop[1]),
        'imbalance_ratio': float(imbalance_ratio)
    },
    'group_sizes': {
        'race': race_counts.to_dict(),
        'sex': sex_counts.to_dict(),
        'age_cat': age_counts.to_dict()
    },
    'data_quality_checks': {
        'missing_values': 'none',
        'duplicates': 'none' if n_duplicates == 0 else f'{n_duplicates} removed',
        'invalid_ranges': 'none detected',
        'target_validation': 'binary (0, 1)'
    },
    'output_files': {
        'cleaned_data': str(output_path.relative_to(project_root)),
        'features': 'data/processed/compas_features.parquet',
        'target': 'data/processed/compas_target.parquet',
        'sensitive': 'data/processed/compas_sensitive.parquet'
    }
}

# Save log
log_path = METADATA_DIR / "compas_cleaning_log.json"
with open(log_path, 'w') as f:
    json.dump(cleaning_log, f, indent=2)

print(f"✓ Saved cleaning log: {log_path}")

✓ Saved cleaning log: /storage/work/szn5432/TabPFN-for-Criminology/data/metadata/compas_cleaning_log.json


### 5.4 Save Filtering Flowchart Data

In [19]:
# Create filtering flowchart table
flowchart_df = pd.DataFrame(filtering_stages)
flowchart_df['cumulative_removed'] = flowchart_df['n_removed'].cumsum()
flowchart_df['remaining_pct'] = (flowchart_df['n_samples'] / n_initial * 100).round(1)

print("\nFiltering flowchart:")
display(flowchart_df)

# Save
flowchart_path = TABLES_DIR / "filtering_flowchart.csv"
flowchart_df.to_csv(flowchart_path, index=False)
print(f"\n✓ Saved flowchart: {flowchart_path}")


Filtering flowchart:


,stage,description,n_samples,n_removed,cumulative_removed,remaining_pct
0,0_raw_data,Raw ProPublica COMPAS data,6172,0,0,100.0



✓ Saved flowchart: /storage/work/szn5432/TabPFN-for-Criminology/results/tables/filtering_flowchart.csv


---

## 6. Summary

### 6.1 Cleaning Summary

In [21]:
print("=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)
print(f"\nDataset: COMPAS (ProPublica)")
print(f"Notebook: 02a_data_cleaning.ipynb")
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n Sample Size:")
print(f"   Initial:  {n_initial:,} samples")
print(f"   Removed:  {n_total_removed:,} samples ({n_total_removed/n_initial*100:.1f}%)")
print(f"   Final:    {n_final:,} samples")

print(f"\n Features:")
print(f"   Features:    {len(feature_cols)}")
print(f"   Target:      {target_col}")
print(f"   Sensitive:   {len(sensitive_cols)} ({', '.join(sensitive_cols)})")

print(f"\n  Class Balance:")
print(f"   Positive (recidivism):     {class_dist[1]:,} ({class_prop[1]:.1%})")
print(f"   Negative (no recidivism):  {class_dist[0]:,} ({class_prop[0]:.1%})")
print(f"   Imbalance ratio:           {imbalance_ratio:.2f}:1")

print(f"\n Data Quality:")
print(f"   Missing values:  None")
print(f"   Duplicates:      None")
print(f"   Invalid ranges:  None detected")
print(f"   All groups:      Adequate size (≥ {MIN_GROUP_SIZE})")

print(f"\n Output Files:")
print(f"   ✓ data/processed/compas_cleaned.parquet")
print(f"   ✓ data/processed/compas_features.parquet")
print(f"   ✓ data/processed/compas_target.parquet")
print(f"   ✓ data/processed/compas_sensitive.parquet")
print(f"   ✓ data/metadata/compas_cleaning_log.json")
print(f"   ✓ results/tables/filtering_flowchart.csv")

print("\n" + "=" * 60)
print("✓ DATA CLEANING COMPLETE")
print("=" * 60)
print("\nNext step: 02b_feature_engineering.ipynb")

DATA CLEANING SUMMARY

Dataset: COMPAS (ProPublica)
Notebook: 02a_data_cleaning.ipynb
Date: 2025-11-09 22:02:45

 Sample Size:
   Initial:  6,172 samples
   Removed:  963 samples (15.6%)
   Final:    5,209 samples

 Features:
   Features:    8
   Target:      two_year_recid
   Sensitive:   3 (race, sex, age_cat)

  Class Balance:
   Positive (recidivism):     2,544 (48.8%)
   Negative (no recidivism):  2,665 (51.2%)
   Imbalance ratio:           1.05:1

 Data Quality:
   Missing values:  None
   Duplicates:      None
   Invalid ranges:  None detected
   All groups:      Adequate size (≥ 50)

 Output Files:
   ✓ data/processed/compas_cleaned.parquet
   ✓ data/processed/compas_features.parquet
   ✓ data/processed/compas_target.parquet
   ✓ data/processed/compas_sensitive.parquet
   ✓ data/metadata/compas_cleaning_log.json
   ✓ results/tables/filtering_flowchart.csv

✓ DATA CLEANING COMPLETE

Next step: 02b_feature_engineering.ipynb


### 6.2 Decisions for Feature Engineering

Based on cleaning results:

**No issues found:**
- ✓ No missing data (complete cases)
- ✓ No duplicates
- ✓ Valid data ranges
- ✓ Adequate group sizes

**Recommendations for next notebook (02b):**
1. **Transformations:** Consider log transform for count features (right-skewed)
2. **Scaling:** Standardize continuous features for logistic regression
3. **Encoding:** One-hot encode categorical variables (race, sex, age_cat)
4. **Class balancing:** Use balanced class weights (imbalance ratio ~1.2:1)

---

**End of Notebook**